# Lane Seven — Demand Forecasting Pipeline
# **v7.4 Production Candidate** 🏭

> **This notebook produces client-ready forecast outputs.**

## What v7.4 adds over v7.2

| Layer | v7.2 | v7.4 |
|-------|------|------|
| Upstream model | StyleCode XGBoost/LightGBM | **same** |
| Calibration | none | **StyleCode-level bias correction (NEW)** |
| Allocation | recency_only | **recency_only (same champion)** |
| Adjustments | v6.2 ADJ_CONFIG | **same** |
| Outputs | experiment CSVs | **production-grade outputs** |

## Why v7.2 recency_only allocation is the production champion

- v7.2 ablation showed recency_only beat baseline on H1 Jan, H3 Jan, H3 Feb
- v7.3 segmented allocation did not clearly beat recency_only
- caps hurt in all experiments → not used
- global smoothing hurt January → not used globally
- v7.4 keeps recency_only and adds calibration to address H3 Feb upstream bias

## Calibration design

- Built from **backtest data only** (MonthStart ≤ 2025-12) → **no leakage**
- Uses CV fold predictions if available; falls back to historical naive bias
- Evidence tiers: STRONG (≥8 obs) or MODERATE (≥4) → factor applied
- Factor clamped to [0.80, 1.20] — no aggressive recalibration
- WEAK or NONE evidence → factor = 1.0 (no change)

## Required inputs

```
data/gold_fact_monthly_demand_v2.csv
data/dim_date.csv
data/dim_product.csv
```

## Production output files

```
outputs/v7_4_production_sku_forecasts.csv         ← MAIN CLIENT DELIVERABLE
outputs/v7_4_stylecode_forecasts_raw_H*.csv
outputs/v7_4_stylecode_forecasts_calibrated_H*.csv
outputs/v7_4_stylecode_calibration_table.csv
outputs/v7_4_holdout_evaluation.csv
outputs/v7_4_error_decomposition.csv
outputs/v7_4_forecast_risk_flags.csv
outputs/v7_4_production_validation_report.csv
outputs/v7_4_vs_prior_versions.csv
```

## Section 0 — Environment setup

In [1]:
import subprocess, sys
packages = ['pandas','numpy','scikit-learn','xgboost','lightgbm','statsmodels','matplotlib','openpyxl']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+packages)
print('All packages installed.')

All packages installed.


In [2]:
import sys, logging
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
DATA_DIR     = NOTEBOOK_DIR / 'data'
OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S', handlers=[logging.StreamHandler(sys.stdout)], force=True,
)
for _noisy in ['lightgbm','xgboost','prophet','cmdstanpy']:
    logging.getLogger(_noisy).setLevel(logging.ERROR)

GOLD_OPTIONS = ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
gold_present = [f for f in GOLD_OPTIONS if (DATA_DIR/f).exists()]
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'  {chr(10003) if gold_present else chr(10007)}  {gold_present[0] if gold_present else "MISSING"}')
print(f'  {chr(10003) if (DATA_DIR/"dim_date.csv").exists() else chr(10007)+" MISSING"}  dim_date.csv')
print(f'  {chr(10003) if (DATA_DIR/"dim_product.csv").exists() else chr(10007)+" MISSING"}  dim_product.csv')
if not gold_present or not (DATA_DIR/'dim_date.csv').exists() or not (DATA_DIR/'dim_product.csv').exists():
    raise FileNotFoundError('Required files missing.')
print('\nSetup complete.')

DATA_DIR   : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.4\data
OUTPUT_DIR : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.4\outputs
  ✓  gold_fact_monthly_demand_v2.csv
  ✓  dim_date.csv
  ✓  dim_product.csv

Setup complete.


## Section 1 — Configuration

In [3]:
from lane7_forecast.forecast_adjustments import get_config

PHASE = 1

# Dynamically detect holdout months from dim_date
_dim_date_check = pd.read_csv(DATA_DIR / "dim_date.csv", parse_dates=["MonthStart"])
_dim_date_check["MonthStart"] = (
    _dim_date_check["MonthStart"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

_candidate_split_cols = [
    "Split",
    "TrainHoldout",
    "DataSplit",
    "DatasetSplit",
    "Set",
    "IsTrain"
]

_holdout_mask = None
_holdout_source_col = None

for _col in _candidate_split_cols:
    if _col not in _dim_date_check.columns:
        continue

    if _col == "IsTrain":
        _vals = _dim_date_check[_col]

        if _vals.dtype == bool:
            _mask = (_vals == False)
        else:
            _mask = _vals.astype(str).str.strip().str.lower().isin(
                ["0", "false", "holdout"]
            )
    else:
        _mask = (
            _dim_date_check[_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .eq("HOLDOUT")
        )

    if _mask.any():
        _holdout_mask = _mask
        _holdout_source_col = _col
        break

if _holdout_mask is None:
    raise ValueError(
        "Could not detect holdout months from dim_date.csv"
    )

HOLDOUT_MONTHS = sorted(
    _dim_date_check.loc[_holdout_mask, "MonthStart"].unique()
)

HOLDOUT_MONTHS = [pd.Timestamp(m) for m in HOLDOUT_MONTHS]

if len(HOLDOUT_MONTHS) == 0:
    raise ValueError("No holdout months detected")

HOLDOUT_START = HOLDOUT_MONTHS[0].strftime("%Y-%m-%d")
N_HOLDOUT_MONTHS = len(HOLDOUT_MONTHS)

print(f"Detected holdout source column: {_holdout_source_col}")
print("Detected holdout months:")
for m in HOLDOUT_MONTHS:
    print(" ", m.strftime("%Y-%m"))

# v6.2 forecast adjustments (unchanged from prior versions)
ADJ_CONFIG = get_config(
    shrink_threshold=1.3, shrink_factor=0.75, recursive_alpha=0.6,
    blend_threshold=0.20, blend_weight=0.7, intermittent_cap_multiplier=2.0,
)

# Allocation parameters (for recency_only)
ALLOC_PARAMS = dict(
    lookback_months=12, min_lookback_months=6,
    w_recent=3, w_mid=2, w_old=1,
)

# Calibration settings
APPLY_CALIBRATION = True   # Set False to run without calibration for comparison

print('=== v7.4 Production Candidate Configuration ===')
print(f'  Phase           : {PHASE}')
print(f'  Holdout months  : {[m.strftime("%Y-%m") for m in HOLDOUT_MONTHS]}')
print(f'  Apply calibration: {APPLY_CALIBRATION}')
print(f'  Allocation method: recency_only (v7.2 champion)')
print()
print('v6.2 Adjustment Config:')
for k, v in ADJ_CONFIG.items(): print(f'  {k:<35}: {v}')

Detected holdout source column: Split
Detected holdout months:
  2026-01
  2026-02
  2026-03
  2026-04
=== v7.4 Production Candidate Configuration ===
  Phase           : 1
  Holdout months  : ['2026-01', '2026-02', '2026-03', '2026-04']
  Apply calibration: True
  Allocation method: recency_only (v7.2 champion)

v6.2 Adjustment Config:
  shrink_threshold                   : 1.3
  shrink_factor                      : 0.75
  recursive_alpha                    : 0.6
  blend_threshold                    : 0.2
  blend_weight                       : 0.7
  intermittent_cap_multiplier        : 2.0
  trailing_window_3                  : 3
  trailing_window_12                 : 12


## Section 2 — Upstream StyleCode Modeling

In [4]:
from lane7_forecast.pipeline import run_v7_prep, run_cv

scode_prep = run_v7_prep(
    data_dir=DATA_DIR, phase=PHASE,
    lookback_months=ALLOC_PARAMS['lookback_months'],
    min_lookback_months=ALLOC_PARAMS['min_lookback_months'],
)

scode_segments = scode_prep['segments']
scode_segments.to_csv(OUTPUT_DIR/'v7_4_stylecode_segments.csv', index=False)

print(f'StyleCode panel: {scode_prep["panel_seg"]["SKU"].nunique():,} StyleCodes')
for seg, n in scode_segments['Segment'].value_counts().items():
    print(f'  {seg:<15} {n:5d}')

13:43:20  INFO      === v7 Step 1: StyleCode-level data prep (phase=1) ===
13:43:20  INFO      Loaded gold_fact_monthly_demand: 119498 rows, 4 cols
13:43:20  INFO      Loaded dim_date: 116 rows, 6 cols
13:43:20  INFO      Loaded dim_product: 3806 rows, 15 cols
13:43:20  WARNING   Optional file not found, skipping: C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.4\data\dim_customer.csv
13:43:20  INFO      Training period end (phase 1): 2025-12
13:43:21  INFO      [v7] STANDALONE (StyleCode level): 468 SKUs, 538 rows, 19,551 units (0.03% of total)
13:43:21  INFO      [v7] StyleCode demand table: 2336 rows, 61 unique StyleCodes, 2017-05 → 2026-04
13:43:21  INFO      Training period end (phase 1): 2025-12
13:43:21  INFO      Rows after date filter (<= 2025-12): 2190
13:43:21  WARNING   Attribute columns missing from demand and no dim_product provided: ['Category', 'StyleCode', 'ColorCode', 'SizeCode', 'StyleColor']. Filling with 'UNKNOWN'.
13:43:21  INFO   

In [5]:
USE_EXISTING_CV = False
_cv_path   = OUTPUT_DIR/'v7_4_cv_results.csv'
_best_path = OUTPUT_DIR/'v7_4_best_models.csv'

v74_bm = {}

if USE_EXISTING_CV and _cv_path.exists() and _best_path.exists():
    print('Loading cached CV results...')
    _cv = pd.read_csv(_cv_path)
    best_models_v74 = pd.read_csv(_best_path)
    for h in [1,3]:
        v74_bm[h] = best_models_v74[best_models_v74['HorizonMonths']==h].copy()
else:
    for h, cfg in {1:dict(n_folds=6,step_months=3,min_train_months=24),
                   3:dict(n_folds=5,step_months=3,min_train_months=24)}.items():
        print(f'\n--- StyleCode CV H={h} ---')
        cv_df, best_df = run_cv(scode_prep, horizon_months=h, **cfg)
        v74_bm[h] = best_df

best_models_v74 = pd.concat([df for df in v74_bm.values() if not df.empty], ignore_index=True)
if not USE_EXISTING_CV:
    best_models_v74.to_csv(OUTPUT_DIR/'v7_4_best_models.csv', index=False)

print('\n=== v7.4 StyleCode Best Models ===')
print(best_models_v74[['Segment','HorizonMonths','BestModel','mean_WMAPE']].round(2).to_string(index=False))


--- StyleCode CV H=1 ---
13:43:32  INFO      === Step 2: CV for horizon=1 months ===
13:43:32  INFO      Horizon 1-month: dropped 156 rows with null required lags (['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3'])
13:43:32  INFO      Features created for horizon=1 — panel shape: (3694, 34)
13:43:32  INFO      CV: segment=REGULAR, horizon=1, models=['XGBoost', 'LightGBM', 'SeasonalNaive'], folds=6
13:43:37  INFO      Trained XGBoost on 1331 rows, 16 features
13:43:43  INFO      Trained LightGBM on 1331 rows, 16 features
13:43:46  INFO      Trained XGBoost on 1407 rows, 16 features
13:43:49  INFO      Trained LightGBM on 1407 rows, 16 features
13:43:51  INFO      Trained XGBoost on 1485 rows, 16 features
13:43:55  INFO      Trained LightGBM on 1485 rows, 16 features
13:43:58  INFO      Trained XGBoost on 1565 rows, 16 features
13:44:02  INFO      Trained LightGBM on 1565 rows, 16 features
13:44:04  INFO      Trained XGBoost on 1646 rows, 16 features
13:44:08  INFO      Trained LightGBM on 

## Section 3 — Load Backtest Predictions for Calibration

The calibration layer requires fold-level CV predictions (backtest only, no holdout).
If these are not available, the calibration falls back to historical naive bias.

> **Leakage guard:** only data with MonthStart ≤ 2025-12 is used for calibration.
> Jan–Feb 2026 actuals are NEVER used to build calibration factors.

In [6]:
# Try to load CV fold predictions from prior v7.x runs.
# Expected columns: [StyleCodeDesc or SKU, MonthStart, HorizonMonths,
#                    ActualUnits, PredictedUnits]
# These are typically written by walk_forward_cv() or run_cv().

# Candidate paths (try in priority order)
_candidate_paths = [
    OUTPUT_DIR / 'v7_4_cv_fold_predictions.csv',
    OUTPUT_DIR / 'v7_3_cv_fold_predictions.csv',
    OUTPUT_DIR / 'v7_2_shared_stylecode_forecasts_H3.csv',  # SC-level, not fold preds
]

backtest_preds_df = None
for p in _candidate_paths:
    if p.exists():
        _df = pd.read_csv(p, parse_dates=['MonthStart'])
        # Only accept if it has both Actual and Predicted columns
        has_actual = any(c in _df.columns for c in ['ActualUnits','Actual'])
        has_pred   = any(c in _df.columns for c in ['PredictedUnits','Predicted'])
        if has_actual and has_pred:
            backtest_preds_df = _df
            print(f'Loaded backtest predictions: {p.name}  ({len(_df):,} rows)')
            break
        else:
            print(f'Skipped {p.name} (missing Actual/Predicted columns)')

if backtest_preds_df is None:
    print('No backtest fold predictions found.')
    print('Calibration will use historical naive bias (fallback path).')
    print('This produces WEAK/NONE evidence — factors will be 1.0.')
    print('To enable CV-based calibration, ensure run_cv() writes fold predictions')
    print('to: outputs/v7_4_cv_fold_predictions.csv')

No backtest fold predictions found.
Calibration will use historical naive bias (fallback path).
This produces WEAK/NONE evidence — factors will be 1.0.
To enable CV-based calibration, ensure run_cv() writes fold predictions
to: outputs/v7_4_cv_fold_predictions.csv


## Section 4 — Generate Raw StyleCode Forecasts

In [7]:
from lane7_forecast.pipeline import run_forecasts
from lane7_forecast.data_prep import _resolve_training_end
from lane7_forecast.data_prep import build_panel
from lane7_forecast.segmentation import segment_skus, attach_segment

tables      = scode_prep['tables']
gold_df     = tables['demand']
dim_prod_df = scode_prep['dim_product']
train_end   = _resolve_training_end(tables['dim_date'], phase=1)
standalone_skus = scode_prep.get('stylecode_standalone_skus', [])

v74_raw_fc  = {}   # horizon → raw StyleCode forecasts
v74_sa_fc   = {}   # horizon → STANDALONE pass-through

for horizon in [3, 1]:
    bm_h = best_models_v74[best_models_v74['HorizonMonths']==horizon]
    if bm_h.empty:
        print(f'H={horizon}: no best model — skipping.')
        continue
    n_pred = N_HOLDOUT_MONTHS

    scode_fc = run_forecasts(
        prep=scode_prep, best_models_df=bm_h, horizon_months=horizon,
        forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
        phase=PHASE, model_version=f'v7.4-raw-H{horizon}',
        output_path=None, append=False, adjustment_config=ADJ_CONFIG,
    )
    v74_raw_fc[horizon] = scode_fc
    scode_fc.to_csv(OUTPUT_DIR/f'v7_4_stylecode_forecasts_raw_H{horizon}.csv', index=False)
    print(f'H={horizon}: {len(scode_fc):,} rows, {scode_fc["Key"].nunique():,} StyleCodes')

    # STANDALONE
    if standalone_skus:
        sa_gold = gold_df[gold_df['SKU'].isin(standalone_skus)].copy()
        if not sa_gold.empty:
            sa_p = build_panel(sa_gold, tables['dim_date'], phase=PHASE, dim_product_df=dim_prod_df)
            sa_s = segment_skus(sa_p)
            sa_prep = {'tables':tables,'panel':sa_p,'segments':sa_s,'panel_seg':attach_segment(sa_p,sa_s)}
            v74_sa_fc[horizon] = run_forecasts(
                prep=sa_prep, best_models_df=bm_h, horizon_months=horizon,
                forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
                phase=PHASE, model_version=f'v7.4-standalone-H{horizon}',
                output_path=None, append=False, adjustment_config=ADJ_CONFIG,
            )

print('\nRaw StyleCode forecasts complete.')

13:44:31  INFO      Training period end (phase 1): 2025-12
13:44:31  INFO      === Step 3: Forecast generation (horizon=3, phase=1) ===
13:44:31  INFO      Horizon 3-month: dropped 590 rows with null required lags (['lag_1', 'lag_3', 'lag_6', 'lag_12', 'rolling_mean_6'])
13:44:31  INFO      Features created for horizon=3 — panel shape: (3260, 34)
13:44:33  INFO      Trained XGBoost on 1485 rows, 19 features
13:44:33  INFO      Retrained XGBoost for segment=REGULAR on 1485 rows
13:44:33  INFO      Generating 3-month forecasts for 52 SKUs, 4 months from 2026-01 (adjustments=enabled)
13:44:36  INFO      Forecast complete: 208 rows, horizon=3
H=3: 208 rows, 52 StyleCodes
13:44:36  INFO      Training period end (phase 1): 2025-12
13:44:36  INFO      Rows after date filter (<= 2025-12): 537
13:44:37  INFO      dim_product joined onto panel. Unknown counts per attr: {'StyleCode': 26877}
13:44:37  INFO      Panel built: 26961 rows, 468 unique SKUs, date range 2017-12 to 2025-12
13:44:38  INFO 

## Section 5 — Calibration Layer

Builds a StyleCode × HorizonMonths calibration factor from backtest data.
**No holdout actuals are used here.** All calibration evidence is from MonthStart ≤ 2025-12.

In [8]:
from lane7_forecast.forecast_calibration_v74 import (
    build_stylecode_calibration_table,
    apply_stylecode_calibration,
    validate_calibration_table,
)

BACKTEST_END = pd.Timestamp('2025-12-01')  # last date allowed for calibration

v74_calib_table = build_stylecode_calibration_table(
    backtest_predictions_df=backtest_preds_df,
    gold_df=gold_df,
    dim_product_df=dim_prod_df,
    backtest_end=BACKTEST_END,
    horizon_months_list=[1, 3],
)

v74_calib_table.to_csv(OUTPUT_DIR/'v7_4_stylecode_calibration_table.csv', index=False)

calib_val = validate_calibration_table(v74_calib_table)

print('=== v7.4 Calibration Table Summary ===')
print(f'  Total rows       : {calib_val["n_rows"]}')
print(f'  Applied          : {calib_val["n_applied"]}')
print(f'  By tier          : {calib_val["n_by_tier"]}')
print(f'  Factors in range : {calib_val["factors_in_range"]}')
print(f'  Duplicate keys   : {calib_val["has_duplicate_keys"]}')
if calib_val['any_warnings']:
    print(f'  Warnings         : {calib_val["any_warnings"]}')
print()
if not v74_calib_table.empty:
    applied_rows = v74_calib_table[v74_calib_table['calibration_applied']]
    print(f'Applied calibration factors (evidence STRONG or MODERATE):')
    print(f'  {len(applied_rows)} StyleCode × Horizon pairs will be adjusted')
    if not applied_rows.empty:
        print()
        print(applied_rows[['StyleCodeDesc','HorizonMonths',
                              'calibration_factor','evidence_tier',
                              'n_observations','raw_bias_ratio',
                              'calibration_source']]
              .sort_values('calibration_factor').head(20).to_string(index=False))
else:
    print('Calibration table is empty — all factors will be 1.0.')

13:44:50  INFO      [v7.4 calib] No CV-based calibration available — using historical fallback only.
13:44:50  INFO      [v7.4 calib] Validation: n_rows=104  n_applied=0  in_range=True  dup=False  tiers={'WEAK': 60, 'NONE': 44}
=== v7.4 Calibration Table Summary ===
  Total rows       : 104
  Applied          : 0
  By tier          : {'WEAK': 60, 'NONE': 44}
  Factors in range : True
  Duplicate keys   : False

Applied calibration factors (evidence STRONG or MODERATE):
  0 StyleCode × Horizon pairs will be adjusted


## Section 6 — Apply Calibration + Production Allocation

In [9]:
from lane7_forecast.allocation_v72 import (
    ALLOCATION_VARIANTS, run_allocation_variant,
)

v74_calib_fc = {}
v74_scol_fc  = {}
v74_sku_fc   = {}

for horizon in [3, 1]:
    if horizon not in v74_raw_fc:
        print(f'H={horizon}: no raw forecast — skipping.')
        continue

    raw_fc = v74_raw_fc[horizon]

    # ── Apply calibration ────────────────────────────────────────────────
    if APPLY_CALIBRATION and not v74_calib_table.empty:
        calib_fc = apply_stylecode_calibration(
            scode_forecasts_df=raw_fc,
            calibration_df=v74_calib_table[v74_calib_table['HorizonMonths']==horizon],
        )
    else:
        calib_fc = raw_fc.copy()
        calib_fc['CalibrationFactor']  = 1.0
        calib_fc['CalibrationApplied'] = False

    calib_fc.to_csv(OUTPUT_DIR/f'v7_4_stylecode_forecasts_calibrated_H{horizon}.csv', index=False)
    v74_calib_fc[horizon] = calib_fc

    n_adj = calib_fc['CalibrationApplied'].sum() if 'CalibrationApplied' in calib_fc.columns else 0
    print(f'H={horizon}: calibration applied to {n_adj}/{len(calib_fc)} rows')

    # ── Production allocation: v7.2 recency_only ─────────────────────────
    sa_fc = v74_sa_fc.get(horizon, None)
    alloc = run_allocation_variant(
        variant_name='recency_only',
        variant_cfg=ALLOCATION_VARIANTS['recency_only'],
        scode_forecasts_df=calib_fc,
        gold_df=gold_df,
        dim_product_df=dim_prod_df,
        train_end=train_end,
        standalone_fc_df=sa_fc,
        **ALLOC_PARAMS,
        smooth_alpha=0.0,
        cap_rel_increase=999.0,   # no caps
    )
    v74_scol_fc[horizon] = alloc['stylecolor_forecasts']
    v74_sku_fc[horizon]  = alloc['sku_forecasts']

    val = alloc['validation']
    print(f'  Validation: sc→scol={val["sc_to_scol_sum_ok"]}  '
          f'scol→sku={val["scol_to_sku_sum_ok"]}  '
          f'no_neg={val["no_negatives"]}  '
          f'SKUs={alloc["sku_forecasts"]["Key"].nunique():,}')
    print()

13:44:50  INFO      [v7.4 calib] Applied calibration: 0 / 208 rows had factor ≠ 1.0
H=3: calibration applied to 0/208 rows
13:44:50  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
13:45:07  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0009)  scol→sku=True (diff=0.0003)  no_neg=True
  Validation: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882

13:45:08  INFO      [v7.4 calib] Applied calibration: 0 / 208 rows had factor ≠ 1.0
H=1: calibration applied to 0/208 rows
13:45:08  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
13:45:21  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0004)  scol→sku=True (diff=0.0003)  no_neg=True
  Validation: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882



## Section 7 — Build Production SKU Forecast Table

In [10]:
from lane7_forecast.production_outputs_v74 import build_production_sku_table

# Combine H=3 and H=1
all_sku_fc = pd.concat(
    [df for df in v74_sku_fc.values() if not df.empty], ignore_index=True
)

prod_sku = build_production_sku_table(
    sku_fc_df=all_sku_fc,
    dim_product_df=dim_prod_df,
    calibration_df=v74_calib_table if not v74_calib_table.empty else None,
    allocation_method='recency_only_v7.2',
)

prod_sku.to_csv(OUTPUT_DIR/'v7_4_production_sku_forecasts.csv', index=False)

print('=== v7.4 Production SKU Forecast Table ===')
print(f'  Rows          : {len(prod_sku):,}')
print(f'  Unique SKUs   : {prod_sku["SKU"].nunique():,}')
print(f'  MonthStart    : {sorted(prod_sku["MonthStart"].unique())}')
print(f'  Horizons      : {sorted(prod_sku["HorizonMonths"].unique())}')
print()
print('Columns:', list(prod_sku.columns))
print()
print('Sample (H=3, first 5 rows):')
print(prod_sku[prod_sku['HorizonMonths']==3].head(5)[[
    'MonthStart','HorizonMonths','SKU','StyleCodeDesc','StyleColorDesc',
    'ForecastUnits','CalibrationApplied','CalibrationFactor','AllocationMethod'
]].to_string(index=False))

13:45:29  INFO      [v7.4] Production SKU table: 31056 rows, 3882 SKUs, 4 months
=== v7.4 Production SKU Forecast Table ===
  Rows          : 31,056
  Unique SKUs   : 3,882
  MonthStart    : [Timestamp('2026-01-01 00:00:00'), Timestamp('2026-02-01 00:00:00'), Timestamp('2026-03-01 00:00:00'), Timestamp('2026-04-01 00:00:00')]
  Horizons      : [np.int64(1), np.int64(3)]

Columns: ['MonthStart', 'HorizonMonths', 'SKU', 'StyleCodeDesc', 'StyleColorDesc', 'SizeDesc', 'ForecastUnits', 'Lower', 'Upper', 'ModelName', 'ModelVersion', 'AllocationMethod', 'CalibrationApplied', 'CalibrationFactor']

Sample (H=3, first 5 rows):
MonthStart  HorizonMonths             SKU StyleCodeDesc StyleColorDesc  ForecastUnits  CalibrationApplied  CalibrationFactor  AllocationMethod
2026-01-01              3     002 Vintage           NaN     STANDALONE            0.0               False                1.0 recency_only_v7.2
2026-01-01              3 002 Vintage Tee           NaN     STANDALONE            0.0    

## Section 8 — Production Validation Report

In [11]:
from lane7_forecast.production_outputs_v74 import build_production_validation_report

# Validate H=3 (primary horizon)
if 3 in v74_calib_fc and 3 in v74_scol_fc and 3 in v74_sku_fc:
    val_report = build_production_validation_report(
        scode_fc=v74_calib_fc[3],
        scol_fc=v74_scol_fc[3],
        sku_fc=v74_sku_fc[3],
        calibration_df=v74_calib_table if not v74_calib_table.empty else None,
    )
    val_report.to_csv(OUTPUT_DIR/'v7_4_production_validation_report.csv', index=False)

    print('=== v7.4 Production Validation Report ===')
    print(val_report.to_string(index=False))
    n_fail = (~val_report['passed']).sum()
    print()
    if n_fail == 0:
        print('✓ ALL CHECKS PASSED — production-ready')
    else:
        print(f'✗ {n_fail} check(s) FAILED — review before client delivery')

13:45:29  INFO      [v7.4] Production validation: 6 checks, 6 passed, 0 failed
=== v7.4 Production Validation Report ===
                                             check  passed                                    detail
             StyleCode totals == StyleColor totals    True                           max_diff=0.0009
                   StyleColor totals == SKU totals    True                           max_diff=0.0003
                         No negative ForecastUnits    True                           0 negative rows
No duplicate (SKU, MonthStart, HorizonMonths) rows    True                          0 duplicate rows
         Required columns present in SKU forecasts    True                               all present
          Calibration factors within allowed range    True min=1.0000, max=1.0000, allowed=[0.8,1.2]

✓ ALL CHECKS PASSED — production-ready


## Section 9 — Holdout Evaluation (Jan–Feb 2026)

In [12]:
from lane7_forecast.production_outputs_v74 import (
    score_holdout, build_error_decomposition,
)

_gpath = next((DATA_DIR/f for f in ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
               if (DATA_DIR/f).exists()), None)
actuals_df = pd.read_csv(_gpath, parse_dates=['MonthStart'])
actuals_df['MonthStart'] = actuals_df['MonthStart'].dt.to_period('M').dt.to_timestamp()
actuals_df['SKU']        = actuals_df['SKU'].astype(str).str.strip()

# Score against holdout actuals (Jan-Feb 2026)
v74_hold_eval, v74_hold_preds = score_holdout(
    sku_fc=prod_sku,
    actuals_df=actuals_df,
    holdout_months=HOLDOUT_MONTHS,
)
v74_hold_eval.to_csv( OUTPUT_DIR/'v7_4_holdout_evaluation.csv',   index=False)
v74_hold_preds.to_csv(OUTPUT_DIR/'v7_4_holdout_predictions.csv',  index=False)

# Error decomposition at all levels
v74_decomp = {}
for horizon in [3, 1]:
    if horizon not in v74_calib_fc: continue
    v74_decomp[horizon] = build_error_decomposition(
        actuals_df=actuals_df, dim_product_df=dim_prod_df,
        scode_fc=v74_calib_fc[horizon],
        scol_fc=v74_scol_fc[horizon],
        sku_fc=v74_sku_fc[horizon],
        holdout_months=HOLDOUT_MONTHS,
    )

decomp_all = pd.concat([df for df in v74_decomp.values() if not df.empty], ignore_index=True)
decomp_all.to_csv(OUTPUT_DIR/'v7_4_error_decomposition.csv', index=False)

print('=== v7.4 HOLDOUT EVALUATION — Detected Holdout Months (SKU level) ===')
print(v74_hold_eval.to_string(index=False))
print()
print('=== ERROR DECOMPOSITION ===')
print(decomp_all.sort_values(['HorizonMonths','MonthStart','Level']).to_string(index=False))

=== v7.4 HOLDOUT EVALUATION — Detected Holdout Months (SKU level) ===
Level  HorizonMonths MonthStart Segment  N_SKUs  TotalActual  TotalPredicted  AbsError   WMAPE
  SKU              1    2026-01     ALL    1976       949368       920523.96 521707.71 54.9532
  SKU              1    2026-02     ALL    2031       833964       993736.99 510798.85 61.2495
  SKU              1    2026-03     ALL    2092       950719      1038071.99 668219.74 70.2857
  SKU              1    2026-04     ALL    2034       808962       990876.04 535176.55 66.1560
  SKU              3    2026-01     ALL    1976       949368       856363.00 498247.79 52.4821
  SKU              3    2026-02     ALL    2031       833964       884205.02 458893.69 55.0256
  SKU              3    2026-03     ALL    2092       950719       944932.88 613097.47 64.4878
  SKU              3    2026-04     ALL    2034       808962       919713.36 504611.70 62.3777

=== ERROR DECOMPOSITION ===
     Level  HorizonMonths MonthStart  TotalAct

## Section 10 — Forecast Risk Flags

In [13]:
from lane7_forecast.production_outputs_v74 import build_forecast_risk_flags

risk_flags = build_forecast_risk_flags(
    sku_fc_df=prod_sku,
    gold_df=gold_df,
    dim_product_df=dim_prod_df,
    calibration_df=v74_calib_table if not v74_calib_table.empty else None,
    holdout_months=HOLDOUT_MONTHS,
)
risk_flags.to_csv(OUTPUT_DIR/'v7_4_forecast_risk_flags.csv', index=False)

print(f'=== v7.4 Forecast Risk Flags: {len(risk_flags):,} total flags across {risk_flags["SKU"].nunique():,} SKUs ===')
print()
print('By flag type:')
print(risk_flags['risk_flag'].value_counts().to_string())
print()
print('By severity:')
print(risk_flags['risk_severity'].value_counts().to_string())
print()
print('HIGH severity flags:')
high = risk_flags[risk_flags['risk_severity']=='HIGH']
if not high.empty:
    print(high[['SKU','StyleCodeDesc','MonthStart','HorizonMonths',
                 'ForecastUnits','risk_flag','reason_code']].head(20).to_string(index=False))
else:
    print('None.')

13:45:36  INFO      [v7.4] Risk flags: 48854 total across 3882 SKUs
=== v7.4 Forecast Risk Flags: 48,854 total flags across 3,882 SKUs ===

By flag type:
risk_flag
calibration_weak_or_missing    31056
sparse_history                 11136
low_recent_demand               3544
intermittent_demand             2872
large_forecast_jump              246

By severity:
risk_severity
LOW       37472
MEDIUM    11136
HIGH        246

HIGH severity flags:
        SKU      StyleCodeDesc MonthStart  HorizonMonths  ForecastUnits           risk_flag                                     reason_code
LS15000BG01 LS15000 Deluxe Tee 2026-01-01              1        29.6442 large_forecast_jump     ForecastUnits=29.6 > 3.0× trailing_mean=6.5
LS15000BG01 LS15000 Deluxe Tee 2026-01-01              3        26.0495 large_forecast_jump     ForecastUnits=26.0 > 3.0× trailing_mean=6.5
LS15000BG01 LS15000 Deluxe Tee 2026-02-01              1        38.6064 large_forecast_jump     ForecastUnits=38.6 > 3.0× trailing_me

## Section 11 — Comparison vs Prior Versions

Loads v7.2 and v7.3 holdout results if available and builds the full comparison table.

In [14]:
from lane7_forecast.production_outputs_v74 import build_version_comparison

# Load prior version results if available
_v72_path = OUTPUT_DIR / 'v7_2_variant_comparison.csv'
_v73_path = OUTPUT_DIR / 'v7_3_vs_v7_2_comparison.csv'

_v72_recency_eval = None
if (OUTPUT_DIR/'H3'/'v7_2_recency_only_holdout_evaluation.csv').exists():
    _v72_recency_eval = pd.read_csv(OUTPUT_DIR/'H3'/'v7_2_recency_only_holdout_evaluation.csv')

_v73_eval = None
if (OUTPUT_DIR/'v7_3_segmented_holdout_evaluation.csv').exists():
    _v73_eval = pd.read_csv(OUTPUT_DIR/'v7_3_segmented_holdout_evaluation.csv')

# Prefer the most complete prior comparison file
_prior_path = _v73_path if _v73_path.exists() else (_v72_path if _v72_path.exists() else None)

comparison = build_version_comparison(
    v74_holdout_eval=v74_hold_eval,
    v74_error_decomp=decomp_all,
    prior_comparison_path=_prior_path,
    v72_recency_holdout_eval=_v72_recency_eval,
    v73_holdout_eval=_v73_eval,
)
comparison.to_csv(OUTPUT_DIR/'v7_4_vs_prior_versions.csv', index=False)

print('=== v7.4 vs PRIOR VERSIONS ===')
print('(ProductionCandidateFlag = True only for v7.4)')
print()
print(comparison[[
    'Variant','H1_Jan_WMAPE','H3_Jan_WMAPE','H3_Feb_WMAPE',
    'StyleCode_H3_Jan_WMAPE','StyleColor_H3_Jan_WMAPE','SKU_H3_Jan_WMAPE',
    'ProductionCandidateFlag'
]].to_string(index=False))
print()
print('Saved: outputs/v7_4_vs_prior_versions.csv')

=== v7.4 vs PRIOR VERSIONS ===
(ProductionCandidateFlag = True only for v7.4)

        Variant  H1_Jan_WMAPE  H3_Jan_WMAPE  H3_Feb_WMAPE  StyleCode_H3_Jan_WMAPE  StyleColor_H3_Jan_WMAPE  SKU_H3_Jan_WMAPE  ProductionCandidateFlag
v7.4_production       54.9532       52.4821       55.0256                 29.3556                  50.3583           52.4821                     True

Saved: outputs/v7_4_vs_prior_versions.csv


## Section 12 — Production Output Summary

In [15]:
print('=== v7.4 Production Candidate — Output File Summary ===')
print()

output_files = [
    ('v7_4_production_sku_forecasts.csv',       'MAIN DELIVERABLE — client-facing SKU forecasts'),
    ('v7_4_stylecode_calibration_table.csv',    'Calibration factors with evidence tiers'),
    ('v7_4_stylecode_forecasts_raw_H3.csv',     'Pre-calibration StyleCode forecasts (H=3)'),
    ('v7_4_stylecode_forecasts_calibrated_H3.csv','Post-calibration StyleCode forecasts (H=3)'),
    ('v7_4_holdout_evaluation.csv',             'WMAPE vs detected holdout actuals'),
    ('v7_4_holdout_predictions.csv',            'Per-row holdout predictions'),
    ('v7_4_error_decomposition.csv',            'WMAPE at StyleCode / StyleColor / SKU'),
    ('v7_4_forecast_risk_flags.csv',            'SKUs needing business review'),
    ('v7_4_production_validation_report.csv',   'Hierarchy reconciliation checks'),
    ('v7_4_vs_prior_versions.csv',              'Full comparison vs v7.2 and v7.3'),
]

all_ok = True
for fname, desc in output_files:
    p = OUTPUT_DIR / fname
    if p.exists():
        n = len(pd.read_csv(p))
        print(f'  ✓  {fname:<55} {n:>7,} rows  {desc}')
    else:
        print(f'  ✗  {fname:<55} NOT FOUND')
        all_ok = False

print()
if all_ok:
    print('ALL PRODUCTION FILES GENERATED — v7.4 is ready for client review.')
else:
    print('Some files not generated. Check for errors in earlier sections.')

=== v7.4 Production Candidate — Output File Summary ===

  ✓  v7_4_production_sku_forecasts.csv                        31,056 rows  MAIN DELIVERABLE — client-facing SKU forecasts
  ✓  v7_4_stylecode_calibration_table.csv                        104 rows  Calibration factors with evidence tiers
  ✓  v7_4_stylecode_forecasts_raw_H3.csv                         208 rows  Pre-calibration StyleCode forecasts (H=3)
  ✓  v7_4_stylecode_forecasts_calibrated_H3.csv                  208 rows  Post-calibration StyleCode forecasts (H=3)
  ✓  v7_4_holdout_evaluation.csv                                   8 rows  WMAPE vs detected holdout actuals
  ✓  v7_4_holdout_predictions.csv                             16,266 rows  Per-row holdout predictions
  ✓  v7_4_error_decomposition.csv                                 24 rows  WMAPE at StyleCode / StyleColor / SKU
  ✓  v7_4_forecast_risk_flags.csv                             48,854 rows  SKUs needing business review
  ✓  v7_4_production_validation_report.csv

---
## Run Order Summary

| Cell | Section | Contents | Required? |
|------|---------|----------|-----------|
| 02 | 0 | pip install | Once |
| 03 | 0 | Path setup | Always |
| **04** | **1** | **ADJ_CONFIG + ALLOC_PARAMS** | **Required** |
| **05** | **2** | **StyleCode panel + CV** | **Required** |
| **06** | **2** | **StyleCode CV** | **Required** |
| **07** | **3** | **Load backtest predictions (optional)** | **Recommended** |
| **08** | **4** | **Raw StyleCode forecasts (H=3, H=1)** | **Required** |
| **09** | **5** | **Build calibration table** | **Required** |
| **10** | **6** | **Apply calibration + allocation** | **Required** |
| **11** | **7** | **Build production SKU table** | **Required** |
| **12** | **8** | **Production validation report** | **Required** |
| **13** | **9** | **Holdout evaluation + error decomp** | **Required** |
| **14** | **10** | **Risk flags** | **Required** |
| **15** | **11** | **Version comparison** | **Required** |
| 16 | 12 | Output summary | Optional |

**Minimum run order:** `03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15`

**Required input files:**

| File | Required? |
|------|-----------|
| `data/gold_fact_monthly_demand_v2.csv` | Yes |
| `data/dim_date.csv` | Yes |
| `data/dim_product.csv` | Yes |
| `outputs/v7_4_cv_fold_predictions.csv` | No (fallback calibration used if absent) |
| `outputs/v7_2_variant_comparison.csv` | No (version comparison only) |
| `outputs/v7_3_vs_v7_2_comparison.csv` | No (version comparison only) |

**Production output files:**

| File | Purpose |
|------|---------|
| `v7_4_production_sku_forecasts.csv` | **Main client deliverable** |
| `v7_4_stylecode_calibration_table.csv` | Calibration audit |
| `v7_4_stylecode_forecasts_raw_H*.csv` | Pre-calibration debug |
| `v7_4_stylecode_forecasts_calibrated_H*.csv` | Post-calibration debug |
| `v7_4_holdout_evaluation.csv` | WMAPE Jan–Feb 2026 |
| `v7_4_error_decomposition.csv` | WMAPE by level |
| `v7_4_forecast_risk_flags.csv` | Business review flags |
| `v7_4_production_validation_report.csv` | Reconciliation checks |
| `v7_4_vs_prior_versions.csv` | Head-to-head comparison |
